# HCDE 530 — Week 5: App reviews exploration

This notebook answers five checklist questions using **`app_reviews_demo.csv`**. Run cells top to bottom (**Shift+Enter**).

**Kernel:** use your Week 5 environment (`.venv` / `week5/.venv`) so `pandas` is available.

## Setup — load the CSV

In [2]:
import pandas as pd
from pathlib import Path

_paths = [Path("app_reviews_demo.csv"), Path("week5/app_reviews_demo.csv")]
_csv = next((p for p in _paths if p.is_file()), None)
if _csv is None:
    raise FileNotFoundError("Place app_reviews_demo.csv in week5/ or run from repo root.")

df = pd.read_csv(_csv)
print(f"Loaded: {_csv.resolve()}")
print(f"pandas {pd.__version__}")

Loaded: /Users/ubahjeyte/hcde530/week5/app_reviews_demo.csv
pandas 3.0.2


---
## 1. What does your dataset look like? `head()`, `info()`

**Answer:** `head()` shows column names, example values, and types at a glance. `info()` lists every column, non-null counts, and dtypes — useful for seeing how many rows you really have per column.

In [3]:
df.head()

,id,app,category,rating,review,date,helpful_votes,verified_purchase,device_type,app_version
0,1,Fieldkit,field research,1,Auto-transcription accuracy on accented speake...,2023-03-31,37,True,mobile,2.5.0
1,2,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-28,12,True,mobile,2.5.3
2,3,Lookback,user research,4,One-click export to Notion is a feature I use ...,2024-03-08,21,True,desktop,5.2.0
3,4,Dovetail,research repository,5,My whole team can comment on the same session ...,2023-12-19,38,True,NaN,2.0.0
4,5,Fieldkit,field research,5,Works offline and syncs when I get back to WiF...,2024-01-23,5,True,desktop,2.5.3


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 500 non-null    int64
 1   app                500 non-null    str  
 2   category           500 non-null    str  
 3   rating             500 non-null    int64
 4   review             500 non-null    str  
 5   date               500 non-null    str  
 6   helpful_votes      500 non-null    int64
 7   verified_purchase  500 non-null    bool 
 8   device_type        437 non-null    str  
 9   app_version        389 non-null    str  
dtypes: bool(1), int64(3), str(6)
memory usage: 35.8 KB


---
## 2. What is the distribution of your most important column?

**Choice:** **`rating`** (1–5) is the main outcome for review data — it tells you how satisfied users are.

**Answer:** The counts below show how many reviews fall in each rating. `normalize=True` adds proportions (0–1).

In [4]:
rating_dist = df["rating"].value_counts().sort_index()
rating_pct = df["rating"].value_counts(normalize=True).sort_index().mul(100).round(1)
summary = pd.DataFrame({"count": rating_dist, "percent": rating_pct})
summary

,count,percent
rating,,
1,29,5.8
2,43,8.6
3,61,12.2
4,160,32.0
5,207,41.4


In [5]:
print("Mean rating:", round(df["rating"].mean(), 2))
print("Median rating:", df["rating"].median())
print(
    "Interpretation: ratings skew high — many 4s and 5s — so the sample is not evenly spread across 1–5."
)

Mean rating: 3.95
Median rating: 4.0
Interpretation: ratings skew high — many 4s and 5s — so the sample is not evenly spread across 1–5.


---
## 3. Filter to a meaningful subset. What is in it?

**Subset:** reviews with **`rating` ≤ 2** (low scores). These rows are the strongest negative signal for product issues.

**Answer:** We report how many rows match, show a sample, and one quick numeric summary (`mean` helpful votes in that slice).

In [6]:
low = df[df["rating"] <= 2]
print(f"Subset size: {len(low)} rows ({len(low) / len(df):.1%} of all reviews)")
low[["app", "category", "rating", "verified_purchase", "device_type", "review"]].head(8)

Subset size: 72 rows (14.4% of all reviews)


,app,category,rating,verified_purchase,device_type,review
0,Fieldkit,field research,1,True,mobile,Auto-transcription accuracy on accented speake...
1,Fieldkit,field research,2,True,mobile,Search results are slow when the repository is...
26,Fieldkit,field research,2,True,mobile,No built-in way to generate a structured debri...
31,Maze,usability testing,1,True,tablet,Session sharing links occasionally expire befo...
32,Lookback,user research,2,True,desktop,Loading large projects takes noticeably longer...
42,Fieldkit,field research,2,True,mobile,Search results are slow when the repository is...
51,Maze,usability testing,1,True,mobile,Session sharing links occasionally expire befo...
74,Miro,collaborative whiteboard,1,True,desktop,I've lost tags twice after a session due to a ...


In [7]:
print("Average helpful_votes in low-rating subset:", round(low["helpful_votes"].mean(), 2))
print("Average helpful_votes overall:", round(df["helpful_votes"].mean(), 2))

Average helpful_votes in low-rating subset: 20.9
Average helpful_votes overall: 23.46


---
## 4. Group by a category and find the average of a numeric column

**Grouping:** **`category`** (product type). **Numeric column:** **`rating`** (also show **`helpful_votes`** for context).

**Answer:** Average rating (and helpful votes) per category — easy comparison across product types.

In [8]:
by_cat = df.groupby("category", observed=True)[["rating", "helpful_votes"]].mean().round(3)
by_cat = by_cat.sort_values("rating", ascending=False)
by_cat

,rating,helpful_votes
category,,
research repository,4.124,24.596
collaborative whiteboard,4.017,25.083
usability testing,4.000,21.968
user research,3.905,21.876
field research,3.674,23.565


---
## 5. Where are the missing values? Are any columns incomplete?

**Answer:** Count missing per column, then express as a percentage of rows. Any column with count **> 0** is incomplete for at least some reviews.

In [9]:
miss = df.isnull().sum()
miss_pct = (df.isnull().mean() * 100).round(2)
missing_report = pd.DataFrame({"missing_count": miss, "missing_pct": miss_pct})
missing_report[missing_report["missing_count"] > 0]

,missing_count,missing_pct
device_type,63,12.6
app_version,111,22.2


In [10]:
incomplete = missing_report[missing_report["missing_count"] > 0].index.tolist()
if incomplete:
    print("Columns with at least one missing value:", ", ".join(incomplete))
else:
    print("No missing values in any column.")

Columns with at least one missing value: device_type, app_version
